# Final project: scaled-up evaluation of open models on KazQAD questions

Research question: does the morphological complexity of Kazakh questions (word-length proxy) affect the quality of answers from open LLMs?
Questions come from `KazQad_part.jsonl` (Maxutov et al., 2024: github.com/akylbekmaxutov/LLM-eval-using-Kazakh). Models run in Colab (T4 GPU).

Run the cells in order. Results are saved after every batch, so after a disconnect just run cells 1-5 again: finished work is skipped.
Before running: Runtime -> Change runtime type -> T4 GPU; secret `HF_TOKEN` added with notebook access on.

In [ ]:
# CELL 1 - setup (about 1-2 minutes)
import os, subprocess, sys
if not os.path.exists("/content/llm-eval-using-kazakh"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/akylbekmaxutov/llm-eval-using-kazakh", "/content/llm-eval-using-kazakh"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "openpyxl"], check=True)
import transformers, torch, pandas, scipy, numpy
print("transformers", transformers.__version__, "| torch", torch.__version__, "| pandas", pandas.__version__, "| scipy", scipy.__version__, "| numpy", numpy.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - change the runtime type")

In [ ]:
# CELL 2 - settings (the only cell you may need to edit)
DATA_FILE = "/content/llm-eval-using-kazakh/Datasets/KazQad_part.jsonl"
OUT_DIR = "/content/results"            # everything is saved here; download it at the end
os.makedirs(OUT_DIR, exist_ok=True)

# selection of questions (same logic as the pilot, filters relaxed to get enough items)
Q_WORDS = (4, 8)          # words in the question
CTX_MAX_WORDS = 150       # words in the passage
N_PER_GROUP = 250         # lowest / highest complexity proxy -> 500 questions
SEED = 42

# models: (short name, Hugging Face id, required?)  A model without access is skipped with a message.
MODELS = [
    ("Qwen3-4B",   "Qwen/Qwen3-4B",            True),
    ("Gemma3-4B",  "google/gemma-3-4b-it",     True),
    ("Til-2B",     "TilQazyna/Til-2B-instruct", False),   # needs access approval; skipped if not yet granted
    # ("Qwen3-1.7B", "Qwen/Qwen3-1.7B",          False),   # fallback third model: remove the leading # to use it
]
MAX_NEW_TOKENS = 96       # one sentence in Kazakh needs more tokens than in English
BATCH_SIZE = 8            # lower to 4 if you get an out-of-memory error
PROMPT = ("Мәтін: {context}\nСұрақ: {question}\n"
          "Мәтін негізінде сұраққа қазақ тілінде бір толық сөйлеммен жауап беріңіз.")   # identical to the pilot

In [ ]:
# CELL 3 - select the questions
import re, json, numpy as np, pandas as pd

def words(t): return re.findall(r"[^\W\d_]+", str(t), flags=re.UNICODE)
def clean(t): return re.sub(r"\s+", " ", re.sub(r"=+", " ", str(t))).strip()

rows = [json.loads(l) for l in open(DATA_FILE, encoding="utf-8")]
df = pd.DataFrame(rows)
df["refs"] = df.answers.map(lambda a: [x for x in a["text"] if str(x).strip()])
df["context_clean"] = df.context.map(clean)
df["question"] = df.question.map(clean)
df["q_words"] = df.question.map(lambda t: len(words(t)))
df["ctx_words"] = df.context_clean.map(lambda t: len(words(t)))
df["proxy"] = df.question.map(lambda t: float(np.mean([len(w) for w in words(t)])) if words(t) else np.nan)
print("items in file:", len(df))
pool = df[df.q_words.between(*Q_WORDS) & (df.ctx_words <= CTX_MAX_WORDS) & (df.refs.map(len) > 0)].drop_duplicates("question")
print("items after filters:", len(pool))
pool = pool.sort_values(["proxy", "id"])
assert len(pool) >= 2 * N_PER_GROUP, "filters leave too few items: relax Q_WORDS or CTX_MAX_WORDS"
low, high = pool.head(N_PER_GROUP).copy(), pool.tail(N_PER_GROUP).copy()
low["condition"], high["condition"] = "low", "high"
sel = pd.concat([low, high]).reset_index(drop=True)
sel["qid"] = [f"K{i+1:03d}" for i in range(len(sel))]
sel["prompt"] = [PROMPT.format(context=c, question=q) for c, q in zip(sel.context_clean, sel.question)]
sel[["qid", "id", "condition", "proxy", "q_words", "ctx_words", "question", "refs", "context_clean"]].to_csv(f"{OUT_DIR}/selected_questions.csv", index=False)
print("\nGroup balance (the groups should differ in proxy, not in length):")
print(sel.groupby("condition")[["proxy", "q_words", "ctx_words"]].agg(["mean", "min", "max"]).round(2).to_string())
from scipy import stats
for c in ("proxy", "q_words", "ctx_words"):
    print(f"Mann-Whitney {c}: p = {stats.mannwhitneyu(low[c], high[c]).pvalue:.4f}")
print("\nExample LOW :", low.question.iloc[0], "| ref:", low.refs.iloc[0])
print("Example HIGH:", high.question.iloc[-1], "| ref:", high.refs.iloc[-1])

In [ ]:
# CELL 4 - generation (the long run: about 15-30 minutes per model on a T4)
import gc, time, torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata
TOKEN = userdata.get("HF_TOKEN")
RESP_FILE = f"{OUT_DIR}/responses.csv"

def load_model(hf_id):
    dtype = torch.bfloat16 if "gemma" in hf_id.lower() else torch.float16   # Gemma is unstable in fp16
    tok = AutoTokenizer.from_pretrained(hf_id, token=TOKEN, padding_side="left")
    try:
        model = AutoModelForCausalLM.from_pretrained(hf_id, token=TOKEN, torch_dtype=dtype, device_map="cuda")
    except Exception as e:                      # some multimodal checkpoints need this class
        print("  AutoModelForCausalLM failed (", type(e).__name__, "), trying AutoModelForImageTextToText")
        from transformers import AutoModelForImageTextToText
        model = AutoModelForImageTextToText.from_pretrained(hf_id, token=TOKEN, torch_dtype=dtype, device_map="cuda")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    return tok, model.eval()

def to_chat(tok, prompt, hf_id):
    kw = {"enable_thinking": False} if "qwen3" in hf_id.lower() else {}          # switch off Qwen3 "thinking" mode
    return tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True, **kw)

def strip_think(t): return re.sub(r"<think>.*?</think>", "", t, flags=re.S).strip()

def generate_batch(tok, model, texts):
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
    return [strip_think(tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True)) for o in out]

done = pd.read_csv(RESP_FILE) if os.path.exists(RESP_FILE) else pd.DataFrame(columns=["qid", "model", "response", "timestamp_utc"])
for name, hf_id, required in MODELS:
    try:
        hf_hub_download(hf_id, "config.json", token=TOKEN)
    except Exception as e:
        msg = f"No access to {hf_id} ({type(e).__name__})."
        if required: raise RuntimeError(msg + " Accept its licence on the model page first.")
        print("SKIPPED:", msg); continue
    todo = sel[~sel.qid.isin(done[done.model == name].qid)].copy()
    print(f"\n=== {name} ({hf_id}): {len(todo)} questions left of {len(sel)}")
    if todo.empty: continue
    tok, model = load_model(hf_id)
    todo["n_tok"] = [len(tok(p).input_ids) for p in todo.prompt]
    todo = todo.sort_values("n_tok")                                   # similar lengths in one batch = faster
    t0 = time.time()
    for i in range(0, len(todo), BATCH_SIZE):
        b = todo.iloc[i:i + BATCH_SIZE]
        resp = generate_batch(tok, model, [to_chat(tok, p, hf_id) for p in b.prompt])
        new = pd.DataFrame({"qid": b.qid.values, "model": name, "response": resp, "timestamp_utc": pd.Timestamp.now("UTC").isoformat()})
        done = pd.concat([done, new], ignore_index=True)
        done.to_csv(RESP_FILE, index=False)                            # saved after every batch
        if (i // BATCH_SIZE) % 5 == 0:
            print(f"  {i + len(b)}/{len(todo)} done, {time.time() - t0:.0f}s | last: {resp[-1][:90]!r}")
    del model, tok; gc.collect(); torch.cuda.empty_cache()
print("\nresponses per model:\n", done.groupby("model").size().to_string())

In [ ]:
# CELL 4b - SMOKE TEST (run BEFORE cell 4 if you want to check one model on 8 questions first)
name, hf_id, _ = MODELS[0]
tok, model = load_model(hf_id)
test = sel.sample(8, random_state=1)
for p, r_, refs in zip(test.prompt, generate_batch(tok, model, [to_chat(tok, p, hf_id) for p in test.prompt]), test.refs):
    print("REF:", refs, "\nANS:", r_, "\n")
del model, tok; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# CELL 5 - automatic scoring
import unicodedata, math
def norm(t):
    t = unicodedata.normalize("NFC", str(t)).lower().replace("ё", "е")
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", t)).strip()
def stem(tok): return tok if len(tok) < 4 else tok[:max(4, math.ceil(0.7 * len(tok)))]

def score(resp, refs):
    """Best over the reference answers: exact-token recall, stem recall (tolerates suffix changes), token F1."""
    rt = norm(resp).split(); rs = [stem(w) for w in rt]; best = dict(recall=0.0, recall_stem=0.0, f1=0.0)
    for ref in refs:
        gt = norm(ref).split()
        if not gt: continue
        common = sum(min(rt.count(w), gt.count(w)) for w in set(gt))
        rec = common / len(gt); prec = common / len(rt) if rt else 0
        gs = [stem(w) for w in gt]
        rec_s = sum(1 for w in gs if w in rs) / len(gs)
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0
        best = dict(recall=max(best["recall"], rec), recall_stem=max(best["recall_stem"], rec_s), f1=max(best["f1"], f1))
    return best

R = pd.read_csv(RESP_FILE).merge(sel[["qid", "condition", "proxy", "q_words", "ctx_words", "question", "refs"]], on="qid")
sc = pd.DataFrame([score(r, refs) for r, refs in zip(R.response, R.refs)])
R = pd.concat([R.reset_index(drop=True), sc], axis=1)
R["resp_words"] = R.response.map(lambda t: len(words(t)))
R["contains_ref"] = [int(any(norm(x) and norm(x) in norm(r) for x in refs)) for r, refs in zip(R.response, R.refs)]
R["cyrillic_ok"] = R.response.map(lambda t: int((lambda L: bool(L) and sum(1 for c in L if "Ѐ" <= c <= "ӿ") / len(L) >= 0.9)([c for c in str(t) if c.isalpha()])))
R["no_info"] = R.response.map(lambda t: int(bool(re.search(r"(ақпарат|мәлімет)[^.]{0,60}(берілмеген|көрсетілмеген|жоқ)|нақты ақпарат|ешқандай", str(t), re.I))))
R.drop(columns="refs").to_csv(f"{OUT_DIR}/responses_scored.csv", index=False)
print("scored responses:", len(R), "| models:", R.model.nunique())
print("empty responses:", int((R.resp_words == 0).sum()), "| not Cyrillic:", int((R.cyrillic_ok == 0).sum()))

In [ ]:
# CELL 5b (OPTIONAL) - check the automatic metrics against the human scores of the pilot (120 responses)
# Upload kazqad_auto_results.csv and kazqad_final_results.csv from the pilot to /content first; skip this cell otherwise.
PA, PF = "/content/kazqad_auto_results.csv", "/content/kazqad_final_results.csv"
if os.path.exists(PA) and os.path.exists(PF):
    a = pd.read_csv(PA, encoding="utf-8-sig"); f = pd.read_csv(PF, encoding="utf-8-sig")
    d = f.merge(a[["qid", "model_key", "response", "ref_answers"]], on=["qid", "model_key"])
    sc2 = pd.DataFrame([score(r, eval(x)) for r, x in zip(d.response, d.ref_answers)]); d = pd.concat([d, sc2], axis=1)
    for c in ["meaning_auto", "recall", "recall_stem", "f1"]:
        print(f"Spearman rho between human meaning and {c}: {stats.spearmanr(d.meaning_human, d[c]).statistic:.2f}")
else:
    print("pilot files not found: cell skipped")

In [ ]:
# CELL 6 - results tables (all in English)
# Why F1: on the 120 pilot responses it had the highest rank correlation with the human meaning score (rho 0.52, against 0.45 for stem recall and 0.37 for the old pilot score)
from scipy import stats
rng = np.random.default_rng(SEED)
def boot_ci(x, n=5000):
    x = np.asarray(x, float); b = [rng.choice(x, len(x)).mean() for _ in range(n)]
    return np.percentile(b, 2.5), np.percentile(b, 97.5)
def rbc(a, b): return 2 * stats.mannwhitneyu(a, b).statistic / (len(a) * len(b)) - 1

METRIC = "f1"      # primary automatic metric: token-level F1 (the metric of Maxutov et al., 2024); alternative: "recall_stem"
print("TABLE 1. Mean", METRIC, "(0-1) by model and complexity group, 95% bootstrap CI")
t1 = []
for m in R.model.unique():
    for g in ("low", "high", "all"):
        x = R[(R.model == m) & ((R.condition == g) | (g == "all"))][METRIC]; lo, hi = boot_ci(x)
        t1.append(dict(model=m, group=g, n=len(x), mean=round(x.mean(), 3), ci=f"[{lo:.3f}, {hi:.3f}]", exact_match_pct=round(100 * R.loc[x.index, "contains_ref"].mean(), 1), words=round(R.loc[x.index, "resp_words"].mean(), 1)))
print(pd.DataFrame(t1).to_string(index=False)); pd.DataFrame(t1).to_csv(f"{OUT_DIR}/table1_main.csv", index=False)

print("\nTABLE 2. Low vs high complexity (per model): difference, 95% CI, Mann-Whitney p, effect size r, Spearman rho (proxy vs score)")
t2 = []
for m in R.model.unique():
    d = R[R.model == m]; a, b = d[d.condition == "low"][METRIC], d[d.condition == "high"][METRIC]
    diffs = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(5000)]
    rho = stats.spearmanr(d.proxy, d[METRIC])
    t2.append(dict(model=m, low=round(a.mean(), 3), high=round(b.mean(), 3), diff=round(a.mean() - b.mean(), 3),
                   ci=f"[{np.percentile(diffs, 2.5):.3f}, {np.percentile(diffs, 97.5):.3f}]", p=round(stats.mannwhitneyu(a, b).pvalue, 4),
                   r=round(rbc(a, b), 3), rho=round(rho.statistic, 3), rho_p=round(rho.pvalue, 4)))
print(pd.DataFrame(t2).to_string(index=False)); pd.DataFrame(t2).to_csv(f"{OUT_DIR}/table2_low_vs_high.csv", index=False)

print("\nTABLE 3. Models on the same questions: Friedman test and paired Wilcoxon tests (Holm-corrected)")
P = R.pivot(index="qid", columns="model", values=METRIC).dropna()
ms = list(P.columns)
if len(ms) >= 3: print("Friedman: chi2 = %.2f, p = %.4g" % stats.friedmanchisquare(*[P[m] for m in ms]))
pairs, raw = [], []
for i in range(len(ms)):
    for j in range(i + 1, len(ms)):
        pairs.append((ms[i], ms[j])); raw.append(stats.wilcoxon(P[ms[i]], P[ms[j]]).pvalue)
order = np.argsort(raw); holm = np.empty(len(raw)); run = 0
for k, idx in enumerate(order): run = max(run, (len(raw) - k) * raw[idx]); holm[idx] = min(1, run)
t3 = pd.DataFrame([dict(pair=f"{a} vs {b}", mean_diff=round((P[a] - P[b]).mean(), 3), wilcoxon_p=round(p, 4), holm_p=round(h, 4)) for (a, b), p, h in zip(pairs, raw, holm)])
print(t3.to_string(index=False)); t3.to_csv(f"{OUT_DIR}/table3_models.csv", index=False)

print("\nTABLE 4. Pooled over models, one value per question (avoids counting a question several times)")
Q = R.groupby(["qid", "condition"], as_index=False)[[METRIC, "proxy"]].mean()
a, b = Q[Q.condition == "low"][METRIC], Q[Q.condition == "high"][METRIC]; rho = stats.spearmanr(Q.proxy, Q[METRIC])
print(f"low {a.mean():.3f}  high {b.mean():.3f}  diff {a.mean() - b.mean():.3f}  Mann-Whitney p = {stats.mannwhitneyu(a, b).pvalue:.4g}  r = {rbc(a, b):.3f}  Spearman rho = {rho.statistic:.3f} (p = {rho.pvalue:.4g})")

print("\nTABLE 5. Response behaviour")
print(R.groupby("model").agg(words_mean=("resp_words", "mean"), no_info_statements=("no_info", "sum"), not_cyrillic=("cyrillic_ok", lambda x: int((x == 0).sum())), empty=("resp_words", lambda x: int((x == 0).sum()))).round(1).to_string())

In [ ]:
# CELL 7 - charts
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ms = list(R.model.unique()); x = np.arange(len(ms))
for j, g in enumerate(("low", "high")):
    means, errs = [], [[], []]
    for m in ms:
        v = R[(R.model == m) & (R.condition == g)][METRIC]; lo, hi = boot_ci(v)
        means.append(v.mean()); errs[0].append(v.mean() - lo); errs[1].append(hi - v.mean())
    ax[0].bar(x + (j - .5) * .35, means, .35, yerr=errs, capsize=3, label=f"{g} complexity", color=["#4C78A8", "#E45756"][j])
ax[0].set_xticks(x); ax[0].set_xticklabels(ms); ax[0].set_ylabel(f"mean {METRIC} (0-1), 95% CI"); ax[0].set_title("Quality by model and complexity group"); ax[0].legend()
for m in ms:
    d = R[R.model == m].copy(); d["bin"] = pd.qcut(d.proxy, 8, duplicates="drop")
    g = d.groupby("bin", observed=True).agg(p=("proxy", "mean"), s=(METRIC, "mean"))
    ax[1].plot(g.p, g.s, marker="o", label=m)
ax[1].set_xlabel("complexity proxy (mean letters per word in the question)"); ax[1].set_ylabel(f"mean {METRIC}"); ax[1].set_title("Quality against complexity (8 bins)"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_results.png", dpi=200); plt.show()

In [ ]:
# CELL 8 - Excel sheet for HUMAN scoring of a random sample (blind: no model name, no group)
N_HUMAN = 120      # about 40 per model; split it between team members if you can (then you can report agreement)
samp = pd.concat([d.sample(min(len(d), N_HUMAN // R.model.nunique()), random_state=SEED) for _, d in R.groupby("model")])
samp = samp.sample(frac=1, random_state=SEED).reset_index(drop=True)
samp["blind_id"] = [f"H{i+1:03d}" for i in range(len(samp))]
sheet = samp[["blind_id", "question", "refs", "response"]].copy(); sheet["refs"] = sheet.refs.map(lambda r: " | ".join(r))
sheet.columns = ["blind_id", "question", "reference_answer", "response"]
sheet["meaning_human (0-2)"] = ""; sheet["grammar_human (0-2)"] = ""
key = samp[["blind_id", "qid", "model", "condition", METRIC]]
with pd.ExcelWriter(f"{OUT_DIR}/human_scoring_sample.xlsx") as w:
    pd.DataFrame({"Rubric": ["Meaning 2: states the correct answer, nothing contradicting it. 1: partly correct, vague, or correct with a wrong extra statement. 0: wrong or missing.",
                             "Grammar 2: no evident errors in case, possessive, agreement, verb forms, vowel harmony, word order. 1: one or two minor errors. 0: three or more errors or hinders understanding.",
                             "Score only the sheet 'Scoring'. Do not open the sheet 'Key' until all scores are entered."]}).to_excel(w, "Rubric", index=False)
    sheet.to_excel(w, "Scoring", index=False); key.to_excel(w, "Key", index=False)
print("saved", f"{OUT_DIR}/human_scoring_sample.xlsx", "with", len(sheet), "responses")

In [ ]:
# CELL 9 - download everything (zip)
import shutil
shutil.make_archive("/content/results_final", "zip", OUT_DIR)
from google.colab import files
files.download("/content/results_final.zip")